# 🛡️ Notebook 2: Fencing Tokens

Whenever a leader is elected, the lock service hands out a **fencing token** — a number that strictly increases on every leadership change. Storage remembers the highest token it has ever seen and **rejects any write with a smaller token**.

Stale leaders simply can't do damage, even if they don't know they're stale.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/split-brain-and-fencing
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
from dataclasses import dataclass, field
from typing import List

class LockService:
    def __init__(self):
        self._next = 0
    def acquire(self, who):
        self._next += 1
        print(f'  granted token {self._next} to {who}')
        return self._next

@dataclass
class Storage:
    highest_seen: int = 0
    log: List[str] = field(default_factory=list)

    def write(self, who, value, token):
        if token < self.highest_seen:
            print(f'  ❌ rejecting write from {who} (token {token} < highest seen {self.highest_seen})')
            return False
        self.highest_seen = token
        self.log.append(f'{who}@{token}: {value}')
        return True

lock = LockService()
store = Storage()

# A acquires lock
tA = lock.acquire('A')
store.write('A', 'config v1', tA)

# A pauses, lock expires, B acquires it (token strictly larger)
tB = lock.acquire('B')
store.write('B', 'config v2', tB)

# A wakes up and tries to write with its old token
store.write('A', 'config v1.1 (STALE)', tA)

print()
print('storage log:')
for entry in store.log:
    print(' ', entry)


Even though A still believes it's leader, storage rejects anything with token `1` after seeing token `2`.

## 🧠 Why fencing is unavoidable

- Leases assume **bounded clocks and bounded GC pauses**. Reality breaks both.
- Without fencing, a stale leader doing a long write *after* its lease expired silently corrupts state.
- Fencing pushes the safety check to the **resource itself** — it doesn't trust callers' beliefs about leadership.

Used by: ZooKeeper (`zxid`), HBase region server epoch, Kubernetes resource versions, Hazelcast, Redis Redlock + fencing.